# Tumor Detection - ResNet (Final Project Skeleton)

**Goal:** Detect tumors in brain MRIs using a transfer-learning ResNet pipeline.

Pipeline mirrors the deliverable spec:
1. Data cleaning & preprocessing
2. Exploratory analysis
3. Image processing with OpenCV
4. Model training with PyTorch (ResNet)
5. Visualization & reporting

Fill in the `TODO` blocks as you work through each phase.

## 0. Environment setup

In [ ]:
import sys, shutil, os

SUPPORT = "/kaggle/input/datasets/ragtxt/brain-tumor-detection-support"
DATA_ROOT = "/kaggle/input/datasets/ragtxt/brain-tumor-roboflow-v1"
WORK = "/kaggle/working"

DATASET = SUPPORT

for f in os.listdir(DATASET):
    shutil.copy(os.path.join(SUPPORT, f), WORK)
sys.path.insert(0, WORK)

os.makedirs(f"{WORK}/models", exist_ok=True)
os.makedirs(f"{WORK}/plots", exist_ok=True)
print("dataset + working dirs ready")

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, random
import pandas as pd
import seaborn as sns
from PIL import Image
import cv2
from preprocess import ImagePreprocessor

torch.manual_seed(36)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Data Cleaning & Preprocessing

**Tasks:**
- Walk the dataset, drop corrupt images.
- Confirm orientation/size consistency.
- Confirm normalization happens in `preprocess.ImagePreprocessor`.

In [ ]:
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

def scan_corrupt(root):
    bad = []
    for d, _, files in os.walk(root):
        for f in files:
            if not f.lower().endswith(IMG_EXTS):
                continue
            p = os.path.join(d, f)
            try:
                with Image.open(p) as im:
                    im.load()         # full decoding 
            except Exception as e:
                bad.append((p, str(e)))
    return bad

bad = scan_corrupt(DATA_ROOT)
print(f"corrupt images: {len(bad)}")
assert len(bad) == 0, f"Found corrupt images: {bad[:5]}"
bad_paths = {p for p, _ in bad}
print(f"corrupt paths: {len(bad)}")
for p, err in bad[:10]:
    print(f" SKIP {p} ({err})")

# TODO: if any bad files remove log bad files 

## 2. Exploratory Analysis

**Tasks:**
- Count images per class per split.
- Plot class distribution.
- Show a grid of sample images per class.

In [ ]:
def class_counts(root):
    out = {}
    for split in ["train", "valid", "test"]:
        sp = os.path.join(root, split)
        if not os.path.isdir(sp):
            continue
        out[split] = {cls: len(os.listdir(os.path.join(sp, cls)))
                      for cls in os.listdir(sp)}
    return out

counts = class_counts(DATA_ROOT)
print(counts)

# wide dict to long dataframe
rows = [(split, cls, n)
           for split, by_cls in counts.items()
           for cls, n in by_cls.items()]
df = pd.DataFrame(rows, columns=["split", "class", "count"])
print(df)

# plot 
plt.figure(figsize=(9, 6))
ax = sns.barplot(data=df, x="split", y="count", hue="class")
for c in ax.containers:
    ax.bar_label(c, fontsize=9)  # bar labels with counts 
plt.title("class distribution per split")
plt.tight_layout()
plt.savefig(f"{WORK}/plots/class_distribution.png", dpi=120)
plt.show()

In [ ]:
# TODO: show 4 tumor + 4 no-tumor sample images in a 2x4 grid

random.seed(36)

N_PER_CLASS = 4
classes = sorted(os.listdir(f"{DATA_ROOT}/train"))    # ['NoTumor', 'Tumor']

fig, axes = plt.subplots(len(classes), N_PER_CLASS, figsize=(12, 6))

for row, cls in enumerate(classes):
    cls_dir = f"{DATA_ROOT}/train/{cls}"
    files = [f for f in os.listdir(cls_dir)
             if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    picks = random.sample(files, N_PER_CLASS)

    for col, fname in enumerate(picks):
        img = Image.open(os.path.join(cls_dir, fname))
        ax = axes[row, col]
        ax.imshow(img, cmap="gray")
        ax.set_title(f"{cls}\n{img.size[0]}×{img.size[1]}", fontsize=9)
        ax.axis("off")

plt.suptitle("Sample images per class (train split)", y=1.02)
plt.tight_layout()
plt.savefig(f"{WORK}/plots/sample_grid.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Image Processing with OpenCV

**Tasks:**
- Grayscale conversion.
- Smoothing (Gaussian / median).
- Thresholding (Otsu).
- Optional: Canny edges + contour overlay for tumor region isolation.

In [ ]:
# pick tumor image from the dataset 

tumor_dir   = f"{DATA_ROOT}/train/Tumor"
sample_path = os.path.join(tumor_dir, sorted(os.listdir(tumor_dir))[0])

# OpenCV pipeline  

img   = cv2.imread(sample_path)
gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur  = cv2.GaussianBlur(gray, (5, 5), 0)
_, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
edges = cv2.Canny(blur, 50, 120)
contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Contour overlay applied in copied data 
overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)

# Display 
panels = [("original - RGB", cv2.cvtColor(img, cv2.COLOR_BGR2RGB)),
          ("grayscale", gray),
          ("gaussian blur", blur),
          ("Otsu threshold", th),
          ("Canny edges", edges),
          ("contour overlay", cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (title, arr) in zip(axes.flat, panels):
    cmap = "gray" if arr.ndim == 2 else None 
    ax.imshow(arr, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.suptitle(f"OpenCV pipeline - {os.path.basename(sample_path)}", y=1.02)
plt.tight_layout()
plt.savefig(f"{WORK}/plots/opencv_pipeline.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"contours found: {len(contours)}")

## 4. Model Training - ResNet (PyTorch)

**Approach:** Transfer learning. ImageNet-pretrained ResNet-18 backbone, replace `fc` head with a 2-class linear layer, train only the head (then optionally unfreeze `layer4` for fine-tuning).

**Tasks:**
- Build train/val/test loaders with `ImagePreprocessor`.
- Define model with replaced head.
- Train with early stopping + best-val checkpoint.
- Plot loss curves.
- (Stretch) repeat with ResNet-50 and compare.

In [ ]:
from torchvision.datasets import ImageFolder

preprocessor = ImagePreprocessor(size=(227, 227))
train_ds = ImageFolder(f"{DATA_ROOT}/train", transform=preprocessor.train_transform)
val_ds   = ImageFolder(f"{DATA_ROOT}/valid", transform=preprocessor.test_transform)
test_ds  = ImageFolder(f"{DATA_ROOT}/test",  transform=preprocessor.test_transform)

BATCH = 32
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=BATCH, shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=BATCH, shuffle=False)

classes = train_ds.classes
print("classes:", classes, "| train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds))

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT)
for p in model.parameters():
    p.requires_grad = False

num_classes = len(classes)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable params:", trainable)

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

EPOCHS = 25
PATIENCE = 7
BEST = f"{WORK}/models/resnet18.pt"

best_val = float("inf")
no_improve = 0
train_losses, val_losses = [], []

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    tl = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        tl += loss.item()
    tl /= len(train_loader)
    train_losses.append(tl)

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            vl += criterion(model(x), y).item()
    vl /= len(val_loader)
    val_losses.append(vl)

    print(f"epoch {epoch+1:02d} | train {tl:.4f} | val {vl:.4f}")

    if vl < best_val:
        best_val = vl
        no_improve = 0
        torch.save(model.state_dict(), BEST)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"early stop @ epoch {epoch+1}")
            break

print("best val loss:", best_val)

In [ ]:
ep = range(1, len(train_losses) + 1)
sns.lineplot(x=ep, y=train_losses, label="train")
sns.lineplot(x=ep, y=val_losses,   label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("ResNet-18 loss")
plt.tight_layout()
plt.savefig(f"{WORK}/plots/loss_curve_resnet18.png")
plt.show()

### 4b. (Optional) Fine-tune `layer4`

After the head converges, unfreeze the last residual block and continue training at a smaller LR.

In [ ]:
# TODO:
# for p in model.layer4.parameters(): p.requires_grad = True
# optimizer = torch.optim.Adam(
#     [p for p in model.parameters() if p.requires_grad], lr=1e-4)
# then re-run the training loop

## 5. Visualization & Reporting

**Tasks:**
- Load best checkpoint, evaluate on test set.
- Print accuracy + classification report.
- Confusion matrix.
- Sample-prediction grid (correct + failures).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

model.load_state_dict(torch.load(BEST, map_location=device))
model.eval()

all_pred, all_true = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        all_pred += model(x).argmax(1).cpu().tolist()
        all_true += y.tolist()

acc = sum(int(p == t) for p, t in zip(all_pred, all_true)) / len(all_true)
print(f"test accuracy: {acc:.4f}")
print(classification_report(all_true, all_pred, target_names=classes))

In [ ]:
cm = confusion_matrix(all_true, all_pred)
ConfusionMatrixDisplay(cm, display_labels=classes).plot(cmap="Blues")
plt.tight_layout()
plt.savefig(f"{WORK}/plots/confusion_resnet18.png")
plt.show()

In [ ]:
# TODO: show 8 test images with predicted vs true labels in a 2x4 grid
# Hint: use test_ds.samples for filepaths, re-open with PIL, denormalize the tensor for display

## 6. Limitations & Future Work

_Fill in for the report / slides._

- Small dataset size and class imbalance.
- MRI artifacts (motion, intensity inhomogeneity) not modeled.
- Binary classification only â€” no tumor localization.
- Generalization to other scanners / hospitals unverified.
- Future: ResNet-50 / EfficientNet baseline, Grad-CAM for interpretability, segmentation with SAM or U-Net.